In [1]:
!py -3.10 -m pip install mediapipe opencv-python
!py -3.10 -m pip install ipykernel
!py -3.10 -m ipykernel install --user --name=python310 --display-name "Python 3.10 (VisoSpeak)"

import cv2
import os
import json
import mediapipe as mp

# Setup paths
video_path = r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data\raw_videos\vid_001.mpg"
video_name = os.path.splitext(os.path.basename(video_path))[0]
video_dir = os.path.join(r"C:\Users\PC\Desktop\phaseB\VisoSpeak\data", video_name)

frames_dir = os.path.join(video_dir, "frames")
mouth_crops_dir = os.path.join(video_dir, "mouth_crops")
metadata_path = os.path.join(video_dir, "metadata.json")

# Load metadata
with open(metadata_path, "r") as f:
    metadata = json.load(f)

# Initialize MediaPipe face mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True)

# Indices of mouth landmarks in MediaPipe's 468-point face mesh
MOUTH_LANDMARKS = list(range(61, 88)) + list(range(291, 318))

for entry in metadata:
    frame_path = os.path.join(frames_dir, entry["frame"])
    image = cv2.imread(frame_path)

    if image is None:
        continue

    h, w = image.shape[:2]
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Run face mesh model
    results = face_mesh.process(rgb_image)

    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0]

        # Extract only mouth landmark points
        mouth_points = [
            (int(landmark.x * w), int(landmark.y * h))
            for i, landmark in enumerate(landmarks.landmark)
            if i in MOUTH_LANDMARKS
        ]

        if mouth_points:
            # Get bounding box around the mouth
            x_coords, y_coords = zip(*mouth_points)
            x1 = max(min(x_coords) - 10, 0)
            y1 = max(min(y_coords) - 10, 0)
            x2 = min(max(x_coords) + 10, w)
            y2 = min(max(y_coords) + 10, h)

            mouth_crop = image[y1:y2, x1:x2]

            # Save mouth crop
            crop_path = os.path.join(mouth_crops_dir, entry["frame"])
            cv2.imwrite(crop_path, mouth_crop)

print(f"✅ Saved MediaPipe-based mouth crops to: {mouth_crops_dir}")



[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: C:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: C:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


Installed kernelspec python310 in C:\Users\PC\AppData\Roaming\jupyter\kernels\python310
✅ Saved MediaPipe-based mouth crops to: C:\Users\PC\Desktop\phaseB\VisoSpeak\data\vid_001\mouth_crops
